In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.signal import cheby2,freqs
import seaborn as sns
import os


In [ ]:
directory_name = "D:\\New folder"

In [ ]:
os.chdir(directory_name)

In [ ]:
os.getcwd()

In [ ]:
# current directory csv files
csvs = [x for x in os.listdir(directory_name) if x.endswith('.csv')]
# stats.csv -> stats
fns = [os.path.splitext(os.path.basename(x))[0] for x in csvs]

dic_csv = {}
for i in range(len(fns)):
    dic_csv[fns[i]] = pd.read_csv(csvs[i])

In [ ]:
for key, df in dic_csv.items():
    parameter_name = df.iloc[:,5].name

In [ ]:
parameter_name

In [ ]:
# Create a new dictionary for cleaned data
clean_dic = {}

# Load cleaned csv files to the dictionary
for key, df in dic_csv.items():
    clean_dic[key] = df[df[parameter_name] > 60]

for key, df in clean_dic.items():
    df.reset_index(inplace=True, drop=True)

In [ ]:
# Generate the list of date elements based on the number of keys
Month_date_list = pd.date_range('2024-04-11 00:00:00.000', freq='1D', periods=len(clean_dic)).strftime('%Y-%m-%d %H:%M:%S.%f').tolist()

In [ ]:
for (key, df), i in zip(clean_dic.items(),Month_date_list) :    
    df['newtime'] = pd.date_range(i, freq='30ms', periods=len(df))

for key, df in clean_dic.items():
    df['newtime'] =pd.to_datetime(df['newtime'])
    
for key, df in clean_dic.items():
    df.set_index(['newtime'], inplace=True)

In [ ]:
total_hrs=0
for key,df in clean_dic.items():
   hrs=np.round((((df.index[-1]-df.index[0]).total_seconds())/3600),decimals=1)
   total_hrs=total_hrs+hrs

In [ ]:
merged_df = pd.concat(clean_dic.values(), ignore_index=True)

merged_df.reset_index(inplace=True, drop=True)

#merged_df['Diaphragm_life'] = [n for n in np.linspace(0,total_hrs,len(merged_df))]

#merged_df.set_index(['Diaphragm_life'], inplace=True)

In [ ]:
merged_df.head(3)

In [ ]:
#merged_df=pd.read_csv(r'D:\\1.5inch Pump Data\\With Vibration Sensor Test\\15_P2_1.5_100-20_01Jun24.csv')

In [ ]:
# Assuming the accelerometer data is in column E (5th column, index 4)
accelerometer_data = merged_df.iloc[:, 3]

In [ ]:
accelerometer_data

In [ ]:
# Step 2: Define the high-pass filter cheby filter
def high_pass_filter_cheby(data, cutoff, fs, order=4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = cheby2(order,normal_cutoff,Wn=1,btype='high', analog=False)
    y = freqs(b, a, data)
    return y

In [ ]:
# Assuming the accelerometer data is in column E (5th column, index 4)
accelerometer_data = merged_df.iloc[:,3]

# Step 2: Define the high-pass filter
def high_pass_filter(data, cutoff, fs, order=4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order,normal_cutoff, btype='high', analog=False)
    y = filtfilt(b, a, data)
    return y

In [ ]:
# Parameters for the high-pass filter
cutoff_frequency = 0.5  # Hz 
sampling_rate = 33.3  # Hz (adjust according to your actual sampling rate)
filtered_data = high_pass_filter_cheby(accelerometer_data, cutoff_frequency, sampling_rate)

In [ ]:
# Parameters for the high-pass filter
cutoff_frequency = 0.5  # Hz 
sampling_rate = 33.3  # Hz (adjust according to your actual sampling rate)
filtered_data = high_pass_filter(accelerometer_data, cutoff_frequency, sampling_rate)

In [ ]:
filtered_data

In [ ]:
Filter_data_df = pd.DataFrame(filtered_data,columns=['Right-Vibration-Data'])

In [ ]:
Filter_data_df.head()

In [ ]:
Filter_data_df['newtime'] = pd.date_range('2024-04-11 00:00:00.000000', freq='30ms', periods=len(Filter_data_df))

Filter_data_df['newtime'] =pd.to_datetime(Filter_data_df['newtime'])

Filter_data_df.set_index(['newtime'], inplace=True)

In [ ]:
# Define the time interval for slicing
start_time = Filter_data_df.index[0]
end_time = Filter_data_df.index[-1]
time_interval = pd.Timedelta(minutes=10)

In [ ]:
freq_amp_dfs_Sf1 = []
result_df = pd.DataFrame()
current_time = start_time
while current_time <= end_time:
            
    next_time = current_time + time_interval
            
    # Slice the DataFrame based on the specified time range
    sliced_df = Filter_data_df.between_time(current_time.time(), next_time.time())
            
    # Perform FFT analysis only if there is data in the sliced DataFrame
    n = len(sliced_df)
    if n > 0:
        t = sliced_df.index
        s = sliced_df['Right-Vibration-Data']

        # Apply high-pass filter
        dt = (t[1] - t[0]).total_seconds()  # Sampling interval in seconds
        
        # FFT
        s -= s.mean()
        fr = np.fft.rfftfreq(n,dt)
        fou = np.fft.rfft(s)
        
        # Create a DataFrame for the current fr,amp
        temp_df = pd.DataFrame({'Frequency': fr,'Amplitude': np.abs(fou)})

        #Filter max apllitude frim temp_df and put it in to df_filter
        df_Filter = temp_df.loc[temp_df['Amplitude'].idxmax()]
        
        #Append coresponding maximum amplitude frequency values in to freq_amp list
        freq_amp_dfs_Sf1.append((df_Filter['Frequency']))

        # Append the DataFrame to the list
        #result_df = pd.concat([result_df, temp_df], ignore_index=True)
        
    else:
        pass
    current_time = next_time    

In [ ]:
#Create Empty Data Frame
Freq_df = pd.DataFrame()
#Add frequency values in to data frame
Freq_df['Frequency'] = freq_amp_dfs_Sf1

In [ ]:
#Reset the data frame
Freq_df.reset_index(inplace=True)

In [ ]:
#Create Diaprhagm liftime
timeline = [n for n in np.linspace(0,total_hrs,len(Freq_df))]
#Insert timeline in to dataframe
Freq_df['Diaphragm_life'] = timeline

In [ ]:
plt.figure(figsize=(30,6))
plt.xticks(np.arange(np.min(Freq_df['Diaphragm_life']),np.max((Freq_df['Diaphragm_life'])),20))
#plt.yticks(np.arange(np.min(Df0_Freq_df['Frequency']),np.max((Df0_Freq_df['Frequency'])),0.5))
plt.plot(Freq_df['Diaphragm_life'],Freq_df['Frequency'],marker = 'o')
plt.xlabel('Hrs')
plt.ylabel('Df0 Freq.')
plt.title('Df0 Vs Diaphragm Lifetime')
#plt.ylim(2.5,6)
#filename = os.path.join(directory_name, f'1.P2_15_10Hrs_Freq_FFT.png')
#plt.savefig(filename)
#plt.close()
plt.show()

In [ ]:
Freq_df.to_csv(f"D:\\1.5inch Pump Data\\With Vibration Sensor Test\\01.T04_S4_P1_15_100-20\\T04_15_S4_280Hrs_Df0.csv")

In [ ]:
# set figure size 
plt.figure( figsize = (30,6)) 
#plt.yticks(np.arange(np.min(Df0_Freq_df['Frequency']),np.max((Df0_Freq_df['Frequency'])),0.5))
sns.regplot(x=Freq_df['Diaphragm_life'], y=Freq_df['Frequency'], lowess=True, line_kws={'color':'red'})
plt.ylim(0,1.5)

In [ ]:
# Step 3: Plot the original and filtered data
plt.figure(figsize=(14, 7))

# Plot original data
plt.subplot(2, 1, 1)
plt.plot(accelerometer_data, label='Original Data')
plt.title('Original Accelerometer Data')
plt.xlabel('Sample Number')
plt.ylabel('Amplitude')
plt.legend()

# Plot filtered data
plt.subplot(2, 1, 2)
plt.plot(filtered_data, label='Filtered Data (1 Hz High-Pass)', color='orange')
plt.title('Filtered Accelerometer Data')
plt.xlabel('Sample Number')
plt.ylabel('Amplitude')
plt.legend()

plt.tight_layout()
plt.show()